In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional-full.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional-names.txt
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-full.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-names.txt
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01"
)

In [3]:
df3 = pd.read_csv(DATA_DIR / "bank-additional-full.csv", sep=";")
df3.drop(columns=['duration'], inplace=True)
#dropped to prevent data leakage
df3.shape

(41188, 20)

In [4]:
#splitting the dataset into X and y and train test and validation splits
y = df3['y'].map({'no': 0, 'yes': 1})
X=df3.drop(columns=['y'])

# train test split

In [5]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Split remaining 30% equally: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)
for name, target in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")
    print(target.value_counts(normalize=True).round(3))


Train
y
0    0.887
1    0.113
Name: proportion, dtype: float64

Validation
y
0    0.887
1    0.113
Name: proportion, dtype: float64

Test
y
0    0.887
1    0.113
Name: proportion, dtype: float64


stratified split preserves target variable distribution across train test and split

In [6]:
import pandas as pd
import numpy as np

def calculate_iv(data, target, bins=10):
    results = []

    for col in data.columns:
        temp = pd.DataFrame({
            'feature': data[col],
            'target': target
        }).copy()

        # Group feature values
        if (
            pd.api.types.is_numeric_dtype(temp['feature'])
            and temp['feature'].nunique() > 10
        ):
            temp['group'] = pd.qcut(
                temp['feature'],
                q=bins,
                duplicates='drop'
            )
        else:
            temp['group'] = (
                temp['feature']
                .astype('string')
                .fillna('MISSING')
            )

        # Aggregate events and non-events
        grouped = temp.groupby(
            'group', observed=True
        )['target'].agg(['count', 'sum'])

        grouped['events'] = grouped['sum']
        grouped['non_events'] = (
            grouped['count'] - grouped['events']
        )

        # Smoothed distributions
        alpha = 0.5

        dist_events = (
            grouped['events'] + alpha
        ) / (
            grouped['events'].sum()
            + alpha * len(grouped)
        )

        dist_non_events = (
            grouped['non_events'] + alpha
        ) / (
            grouped['non_events'].sum()
            + alpha * len(grouped)
        )

        # WOE and IV
        woe = np.log(dist_non_events / dist_events)

        iv = (
            (dist_non_events - dist_events) * woe
        ).sum()

        results.append({
            'Feature': col,
            'IV': iv
        })

    return pd.DataFrame(results).sort_values(
        'IV', ascending=False
    ).reset_index(drop=True)

In [7]:
iv_results = calculate_iv(X_train, y_train)

print(iv_results.to_string(index=False))

       Feature       IV
   nr.employed 1.136348
  emp.var.rate 1.030894
     euribor3m 1.012311
 cons.conf.idx 0.834102
      poutcome 0.547386
         month 0.472781
cons.price.idx 0.413606
      previous 0.356802
       contact 0.243057
           job 0.177166
           age 0.138384
       default 0.122450
     education 0.051751
      campaign 0.050784
       marital 0.031153
   day_of_week 0.003397
       housing 0.000607
          loan 0.000054
         pdays 0.000000


In [8]:
print('nr.employed', df3['nr.employed'].nunique())
print('emp.var.rate', df3['emp.var.rate'].nunique())
print('euribor3m', df3['euribor3m'].nunique())
print('cons.conf.idx', df3['cons.conf.idx'].nunique())
print("poutcome", df3['poutcome'].nunique())

nr.employed 11
emp.var.rate 10
euribor3m 316
cons.conf.idx 26
poutcome 3


In [9]:
for b in [5, 10, 20]:
    result = calculate_iv(
        X_train[['euribor3m']],
        y_train,
        bins=b
    )

    print(b, result)

5      Feature        IV
0  euribor3m  0.785832
10      Feature        IV
0  euribor3m  1.012311
20      Feature        IV
0  euribor3m  1.109206


Information value shoots up: 1) actual predictive value 2) leakage [is the feature available at the time of decision making] 3) binning strategy [more bins more event and non event distiction] 

# Encoding


In [10]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on train; transform validation and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (28831, 52)
Validation: (6178, 52)
Test: (6179, 52)
